# CogVideoX-5B I2V on Kaggle

在 Kaggle T4 GPU 上直接跑 CogVideoX 推理，无需 Web 服务/隧道。

## ⚡ 使用前准备

**1. 打开 GPU（重要！）**
- 点右侧面板 **Settings** → **Accelerator** → 选 **GPU T4 x2**（有 32GB 显存）
- 如果 T4x2 不可用，选 **GPU T4 x1**（16GB，需要降低参数）
- 选 **Internet** → **On**

**2. 上传模板图**
- 点右侧 **+ Add Data** → 选你的 `character_template.png` 上传
- 上传后记下路径，稍后填入 Cell 3
  - 通常是 `/kaggle/input/你的数据集名/character_template.png`
  - 或者点文件右侧的 📋 复制路径

## 1. 装依赖

In [ ]:
!pip install -q diffusers transformers accelerate torchvision opencv-python pillow

## 2. 检查 GPU

In [ ]:
!nvidia-smi

## 3. 设置参数

修改 `IMAGE_PATH` 为你的模板图路径（就是你上传后复制的那条路径）。

**如果不确定路径**：执行下面 cell 前，先运行这行：
```python
import subprocess; subprocess.run(['find', '/kaggle/input/', '-name', '*.png'])
```

In [ ]:
# ==== 请修改这些 ====
IMAGE_PATH = "/kaggle/input/cogvideo-template/character_template.png"  # ← 改成你的路径
OUTPUT_DIR = "/kaggle/working/"
PROMPT = "a cute anime mother holding her baby, gentle horizontal sway, warm pastel colors, simple background"
NEGATIVE_PROMPT = "ugly, blurry, deformed, text, watermark, complex background, realistic"
SEED = 42
NUM_FRAMES = 25          # T4x2(32GB) 可用 25 帧; T4x1(16GB) 建议 16 帧
NUM_STEPS = 20
GUIDANCE_SCALE = 6.0
WIDTH = 480
HEIGHT = 720
# =====================

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4. 加载模型

In [ ]:
import torch
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image
import time

print("Loading model (first run downloads ~2GB)...")
t0 = time.time()

pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    torch_dtype=torch.bfloat16
)
pipe.enable_sequential_cpu_offload()
pipe.enable_attention_slicing(slice_size="auto")
pipe.vae.enable_tiling()

print(f"Done in {time.time()-t0:.1f}s")

## 5. 跑推理

In [ ]:
print(f"Loading image: {IMAGE_PATH}")
image = Image.open(IMAGE_PATH).convert("RGB")
print(f"Original size: {image.size}")

image = image.resize((WIDTH, HEIGHT), Image.LANCZOS)
print(f"Resized to: {image.size}")

print(f"\nGenerating {NUM_FRAMES} frames...")
t0 = time.time()

generator = torch.Generator(device="cpu").manual_seed(SEED)
video_frames = pipe(
    prompt=PROMPT,
    image=image,
    num_frames=NUM_FRAMES,
    guidance_scale=GUIDANCE_SCALE,
    num_inference_steps=NUM_STEPS,
    generator=generator,
    negative_prompt=NEGATIVE_PROMPT,
).frames[0]

print(f"Done in {time.time()-t0:.1f}s ({len(video_frames)} frames)")

## 6. 保存 + 预览

In [ ]:
output_file = os.path.join(OUTPUT_DIR, f"cogvideo_output.mp4")
export_to_video(video_frames, output_file, fps=8)
print(f"Saved: {output_file}")
print(f"Size: {os.path.getsize(output_file)/1024/1024:.1f} MB")

# 预览（Kaggle 支持 HTML5 video 播放）
from IPython.display import Video, display
display(Video(output_file, width=480))